# 01 — Data Prep & Sampling

Reconciles `points.csv` ∩ `metadata.csv` ∩ actual front-view images present,
for both classes — this three-way join is necessary because `points.csv`
still contains ids without a corresponding SVI file. Then assigns repeated
spatial-block CV folds across **both classes jointly**, so fold membership
is consistent once training starts.

Output: `interim/reconciled_points.parquet` — the manifest every later
notebook (`02`, `04`) iterates over.

In [ ]:
# ── Clone/update repo, mount Drive, load config ─────────────────────────
REPO_URL = "https://github.com/AditPradana36/crash-dualgraph.git"
REPO_DIR = "/content/crash-dualgraph"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(f"{REPO_DIR}/src")

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q pyyaml tqdm pandas geopandas shapely scikit-learn seaborn

In [ ]:
import yaml
import numpy as np
from pathlib import Path

with open(f"{REPO_DIR}/configs/paths.yaml") as f:
    paths_cfg = yaml.safe_load(f)

BASE_DIR = Path(paths_cfg["base_dir"])
INTERIM_DIR = Path(paths_cfg["interim_dir"])
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

RECONCILED_OUT = INTERIM_DIR / "reconciled_points.parquet"
FORCE_RECOMPUTE = False  # set True to rebuild even if the output already exists

print(f"Base dir:    {BASE_DIR}")
print(f"Output:      {RECONCILED_OUT}")
print(f"Expect ~{paths_cfg['approx_n_positive']}+ positive, ~{paths_cfg['approx_n_negative']}+ negative")

In [ ]:
# ── Skip everything below if already done (checkpoint pattern) ──────────
if RECONCILED_OUT.exists() and not FORCE_RECOMPUTE:
    import pandas as pd
    reconciled = pd.read_parquet(RECONCILED_OUT)
    print(f"✅ Found existing reconciled output ({len(reconciled)} rows) — loaded, skipping recomputation.")
    print("   Set FORCE_RECOMPUTE = True above and re-run if you need to rebuild it.")
else:
    print("No existing output found (or FORCE_RECOMPUTE=True) — running full reconciliation below.")

In [ ]:
# ── Reconciliation logic ─────────────────────────────────────────────────
import re
import pandas as pd
from tqdm.auto import tqdm

IMG_PATTERN = re.compile(r"^(\d+)_dashcamfov90\.(jpg|jpeg|png)$", re.IGNORECASE)

def extract_id_from_filename(fname: str):
    m = IMG_PATTERN.match(fname)
    return int(m.group(1)) if m else None

def reconcile_class(cls_name, points_csv, metadata_csv, front_view_dir):
    pts = pd.read_csv(points_csv)
    meta = pd.read_csv(metadata_csv)

    image_ids = {}
    for p in Path(front_view_dir).iterdir():
        _id = extract_id_from_filename(p.name)
        if _id is not None:
            image_ids[_id] = str(p)

    ids_points = set(pts["id"])
    ids_meta = set(meta["id"])
    ids_images = set(image_ids.keys())

    ids_pts_not_meta = ids_points - ids_meta
    ids_meta_not_pts = ids_meta - ids_points
    ids_pm = ids_points & ids_meta
    ids_pm_not_img = ids_pm - ids_images
    ids_final = ids_pm & ids_images

    print(f"[{cls_name}]")
    print(f"  points.csv={len(ids_points)}  metadata.csv={len(ids_meta)}  images_on_disk={len(ids_images)}")
    print(f"  points ∩ metadata = {len(ids_pm)}  "
          f"(dropped: {len(ids_pts_not_meta)} points-only, {len(ids_meta_not_pts)} metadata-only)")
    print(f"  (points ∩ metadata) ∩ images = {len(ids_final)}  "
          f"(further dropped, no image file: {len(ids_pm_not_img)})")
    if ids_pm_not_img:
        sample = sorted(list(ids_pm_not_img))[:10]
        print(f"  example ids missing an image (first 10): {sample}")
    print()

    merged = pts.merge(meta, on="id", how="inner")
    merged = merged[merged["id"].isin(ids_images)].copy()
    merged["image_path"] = merged["id"].map(image_ids)
    merged["class"] = cls_name
    merged["label"] = 1 if cls_name == "positive" else 0
    merged["point_id"] = merged["class"] + "_" + merged["id"].astype(str)

    # Informational only — metadata_outofrange_removed.csv implies distance
    # filtering already happened upstream; not re-applied here.
    if "status" in merged.columns:
        n_not_ok = (merged["status"] != "ok").sum()
        if n_not_ok:
            print(f"  ⚠️  {n_not_ok} reconciled rows have status != 'ok' — "
                  f"unexpected given the filename, worth a manual check.")
    return merged

In [ ]:
if not (RECONCILED_OUT.exists() and not FORCE_RECOMPUTE):
    class_specs = [
        ("positive", paths_cfg["positive_points_csv"], paths_cfg["positive_metadata_csv"], paths_cfg["positive_front_view_dir"]),
        ("negative", paths_cfg["negative_points_csv"], paths_cfg["negative_metadata_csv"], paths_cfg["negative_front_view_dir"]),
    ]

    reconciled_parts = []
    for cls_name, pts_csv, meta_csv, fv_dir in tqdm(class_specs, desc="Reconciling classes"):
        reconciled_parts.append(reconcile_class(cls_name, pts_csv, meta_csv, fv_dir))

    reconciled = pd.concat(reconciled_parts, ignore_index=True)
    print(f"Combined reconciled dataset: {len(reconciled)} rows "
          f"({(reconciled['label']==1).sum()} positive, {(reconciled['label']==0).sum()} negative)")

In [ ]:
# ── Assign repeated spatial-block CV folds (both classes jointly) ───────
# Positions are input_lat/input_lon (the true incident coordinate) —
# pano_lat/pano_lon is the panorama's own position, kept only as metadata.
if not (RECONCILED_OUT.exists() and not FORCE_RECOMPUTE):
    import geopandas as gpd
    from sklearn.cluster import KMeans

    with open(f"{REPO_DIR}/configs/eval.yaml") as f:
        eval_cfg = yaml.safe_load(f)

    K_FOLDS = eval_cfg["k_folds"]
    REPEATS = eval_cfg["repeats"]

    boundary = gpd.read_file(paths_cfg["boundary_geojson"])
    if boundary.crs is None:
        boundary = boundary.set_crs(epsg=4326)
    utm_crs = boundary.estimate_utm_crs()
    print(f"Using UTM CRS for spatial clustering: {utm_crs}")

    pts_gdf = gpd.GeoDataFrame(
        reconciled,
        geometry=gpd.points_from_xy(reconciled["input_lon"], reconciled["input_lat"]),
        crs="EPSG:4326",
    ).to_crs(utm_crs)

    coords = np.column_stack([pts_gdf.geometry.x.values, pts_gdf.geometry.y.values])

    for r in tqdm(range(REPEATS), desc="Assigning spatial folds"):
        km = KMeans(n_clusters=K_FOLDS, random_state=42 + r, n_init=10)
        reconciled[f"fold_rep{r}"] = km.fit_predict(coords)

    print(f"Assigned {REPEATS} repeat(s) x {K_FOLDS} spatial folds.")

In [ ]:
# ── QC: class balance per fold — spatial blocking can produce uneven
#    pos:neg ratios per fold (hotspot-driven positives vs. hotspot-excluded
#    negatives) — worth seeing explicitly, not assuming it's balanced. ────
if not (RECONCILED_OUT.exists() and not FORCE_RECOMPUTE):
    for r in range(REPEATS):
        print(f"--- repeat {r} — class balance per fold ---")
        display(reconciled.groupby(f"fold_rep{r}")["label"]
                .agg(["count", "sum"]).rename(columns={"sum": "n_positive"}))

In [ ]:
# ── Visualize spatial fold layout ────────────────────────────────────────
if not (RECONCILED_OUT.exists() and not FORCE_RECOMPUTE):
    import seaborn as sns
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, REPEATS, figsize=(6 * REPEATS, 6))
    if REPEATS == 1:
        axes = [axes]
    for r, ax in enumerate(axes):
        boundary.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=1)
        sns.scatterplot(
            data=reconciled, x="input_lon", y="input_lat",
            hue=f"fold_rep{r}", style="label", palette="tab10",
            s=40, ax=ax,
        )
        ax.set_title(f"Spatial folds — repeat {r}")
    plt.tight_layout()
    plt.savefig(f"{INTERIM_DIR}/qc_spatial_folds.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ── Save checkpoint ───────────────────────────────────────────────────────
if not (RECONCILED_OUT.exists() and not FORCE_RECOMPUTE):
    reconciled.to_parquet(RECONCILED_OUT, index=False)
    print(f"✅ Saved {len(reconciled)} reconciled points to {RECONCILED_OUT}")

print()
print(reconciled[["point_id", "class", "label", "highway", "input_lat", "input_lon"]].head(10))
print()
print("Next: 02_svi_segmentation.ipynb")